In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("FoodDeliveryAnalytics")
    .config(
        "spark.jars.packages",
        "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.1"
    )
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Spark session created.")
print("Spark version:", spark.version)

Spark session created.
Spark version: 3.5.9


In [2]:
from pyspark.sql.types import (
    StructType, StructField, LongType,
    IntegerType, StringType
)

schema = StructType([
    StructField("order_id", LongType(), True),
    StructField("restaurant_id", IntegerType(), True),
    StructField("delivery_area", StringType(), True),
    StructField("cuisine", StringType(), True),
    StructField("status", StringType(), True),
    StructField("estimated_delivery_time", IntegerType(), True),
    StructField("customer", StringType(), True),
    StructField("event_time", StringType(), True)
])

In [3]:
from pyspark.sql.functions import (
    col, from_json, to_timestamp, window,
    count, when, sum as spark_sum
)

TOPIC = "food-orders"
BROKER = "localhost:9092"

raw_stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", BROKER)
    .option("subscribe", TOPIC)
    .option("startingOffsets", "latest")
    .load()
)

json_stream = raw_stream.select(
    col("value").cast("string").alias("json_value")
)

orders = (
    json_stream
    .select(from_json(col("json_value"), schema).alias("data"))
    .select("data.*")
    .withColumn("event_time", to_timestamp("event_time"))
    .filter(
        col("order_id").isNotNull()
        & col("delivery_area").isNotNull()
        & col("status").isNotNull()
        & col("event_time").isNotNull()
    )
)

orders.printSchema()

root
 |-- order_id: long (nullable = true)
 |-- restaurant_id: integer (nullable = true)
 |-- delivery_area: string (nullable = true)
 |-- cuisine: string (nullable = true)
 |-- status: string (nullable = true)
 |-- estimated_delivery_time: integer (nullable = true)
 |-- customer: string (nullable = true)
 |-- event_time: timestamp (nullable = true)



In [15]:
#question 1
status_counts = (
    orders
    .withWatermark("event_time", "2 minutes")
    .groupBy(
        window(col("event_time"), "1 minute"),
        col("delivery_area"),
        col("status")
    )
    .agg(count("*").alias("order_count"))
)

query1 = (
    status_counts.writeStream
    .outputMode("update")
    .format("console")
    .option("truncate", "false")
    .option("checkpointLocation", "./checkpoints/status_counts")
    .trigger(processingTime="10 seconds")
    .start()
)

print("Task 1 started: order counts by status and area.")

Task 1 started: order counts by status and area.


In [13]:
#question 2
DELAYED_ORDER_THRESHOLD = 3
CANCELLED_ORDER_THRESHOLD = 2

problem_orders = (
    orders
    .withWatermark("event_time", "2 minutes")
    .filter(col("status").isin("delayed", "cancelled"))
    .groupBy(
        window(col("event_time"), "1 minute"),
        col("delivery_area")
    )
    .agg(
        spark_sum(
            when(col("status") == "delayed", 1).otherwise(0)
        ).alias("delayed_count"),
        spark_sum(
            when(col("status") == "cancelled", 1).otherwise(0)
        ).alias("cancelled_count")
    )
    .filter(
        (col("delayed_count") >= DELAYED_ORDER_THRESHOLD)
        | (col("cancelled_count") >= CANCELLED_ORDER_THRESHOLD)
    )
)

query2 = (
    problem_orders.writeStream
    .outputMode("update")
    .format("console")
    .option("truncate", "false")
    .option("checkpointLocation", "./checkpoints/problem_areas")
    .trigger(processingTime="10 seconds")
    .start()
)

print("Task 2 started: delayed and cancelled order alerts.")

Task 2 started: delayed and cancelled order alerts.


In [9]:
#question 3
cuisine_counts = (
    orders
    .withWatermark("event_time", "3 minutes")
    .groupBy(
        window(col("event_time"), "2 minutes"),
        col("delivery_area"),
        col("cuisine")
    )
    .agg(count("*").alias("order_count"))
)

query3 = (
    cuisine_counts.writeStream
    .outputMode("update")
    .format("console")
    .option("truncate", "false")
    .option("checkpointLocation", "./checkpoints/cuisine_counts")
    .trigger(processingTime="10 seconds")
    .start()
)

print("Task 3 started: cuisine counts by area.")

Task 3 started: cuisine counts by area.


In [11]:
#question 4
DEMAND_THRESHOLD = 5

area_demand = (
    orders
    .withWatermark("event_time", "3 minutes")
    .groupBy(
        window(col("event_time"), "2 minutes"),
        col("delivery_area")
    )
    .agg(count("*").alias("demand_count"))
    .filter(col("demand_count") >= DEMAND_THRESHOLD)
)

query4 = (
    area_demand.writeStream
    .outputMode("update")
    .format("console")
    .option("truncate", "false")
    .option("checkpointLocation", "./checkpoints/area_demand")
    .trigger(processingTime="10 seconds")
    .start()
)

print("Task 4 started: high-demand area alerts.")

Task 4 started: high-demand area alerts.


In [16]:
for query in spark.streams.active:
    query.stop()

print("All streaming queries stopped.")

All streaming queries stopped.
